# Simulation: Generate All Noise Variants

This notebook generates simulated clonal data with two types of noise applied at varying levels (0% to 100%, every 2%) for all 12 metaclones.

## Noise Types

### 1. Additive Gaussian (Normal) Noise
- Standard deviation = `deterministic_value × (P / 100)`
- At 0%: no noise (deterministic baseline)
- At 100%: std dev equals the deterministic value itself
- Values are rounded to integers and clipped at 0

### 2. Uniform Redistribution Noise
- For each timepoint block (13 cell types):
  1. Compute total cell count across all 13 columns
  2. Randomly select P% of total cells from occupied columns (weighted by count)
  3. Redistribute those cells uniformly across random columns
- At 0%: no redistribution (identical to baseline)
- At 100%: completely random distribution (total count preserved, but profile is random)
- **Key property**: Total cell count per timepoint is preserved

## Output Structure
```
All_Noise/
  MC0/                    # Metaclone 1 (0-indexed directory)
    dev_0_19/             # Noise levels 0-18%
      mc_1_gaussian_0.csv
      mc_1_uniform_0.csv
      mc_1_gaussian_2.csv
      mc_1_uniform_2.csv
      ...
    dev_20_39/            # Noise levels 20-38%
    dev_40_59/            # Noise levels 40-58%
    dev_60_79/            # Noise levels 60-78%
    dev_80_100/           # Noise levels 80-100%
  MC1/                    # Metaclone 2
    ...
  MC11/                   # Metaclone 12
    ...
```


In [1]:
import numpy as np
import os

# =========================================================
# Configuration
# =========================================================

STANDARD_DIR = 'simulated_data/Standard'
OUTPUT_DIR = 'simulated_data/All_Noise'

# Noise levels: 0, 2, 4, ..., 100 (51 levels)
NOISE_LEVELS = list(range(0, 101, 2))

N_CELL_TYPES = 13
N_TIMEPOINTS = 4

# Replicate counts per metaclone (matching the standard dataset)
REPLICATE_COUNTS = {
    1: 15,  2: 30,  3: 50,  4: 40,
    5: 72,  6: 56,  7: 88,  8: 20,
    9: 37, 10: 53, 11: 26, 12: 30
}

print(f"Noise levels: {len(NOISE_LEVELS)} levels from {NOISE_LEVELS[0]}% to {NOISE_LEVELS[-1]}%")
print(f"Metaclones: 12")
print(f"Noise types: Gaussian + Uniform")
print(f"Total files to generate: {12 * len(NOISE_LEVELS) * 2} = {12} MC × {len(NOISE_LEVELS)} levels × 2 types")


Noise levels: 51 levels from 0% to 100%
Metaclones: 12
Noise types: Gaussian + Uniform
Total files to generate: 1224 = 12 MC × 51 levels × 2 types


In [2]:
# =========================================================
# Helper Functions
# =========================================================

def get_dev_folder(pct):
    """Return the dev_X_Y subfolder name for a given noise percentage."""
    if pct <= 19:
        return "dev_0_19"
    elif pct <= 39:
        return "dev_20_39"
    elif pct <= 59:
        return "dev_40_59"
    elif pct <= 79:
        return "dev_60_79"
    else:
        return "dev_80_100"


def load_deterministic_baseline(meta_id):
    """
    Load standard CSV data and recover the deterministic baseline
    by computing the median across noise replicates.
    
    The standard dataset was generated with 20% Gaussian noise.
    The median across replicates converges to the true deterministic
    ODE solution for each metaclone.
    """
    csv_path = os.path.join(STANDARD_DIR, f'simulated_clones_meta{meta_id}.csv')
    data = np.loadtxt(csv_path, delimiter=',')
    
    # Recover deterministic baseline via median (robust to noise outliers)
    baseline = np.median(data, axis=0)
    
    # Round to clean integer counts and clip negatives
    baseline = np.round(baseline)
    baseline[baseline < 0] = 0
    
    return baseline


def apply_gaussian_noise(deterministic, pct, num_replicates, rng_seed=42):
    """
    Apply additive Gaussian (normal) noise to the deterministic baseline.
    
    For each element:
        dev = deterministic_value * (pct / 100)
        noise ~ N(0, dev)
        noisy_value = round(deterministic + noise)
        noisy_value = max(0, noisy_value)
    
    Parameters
    ----------
    deterministic : np.ndarray, shape (52,)
        Flattened deterministic baseline (4 timepoints × 13 cell types).
    pct : int
        Noise percentage (0-100).
    num_replicates : int
        Number of noisy replicates to generate.
    rng_seed : int
        Random seed for reproducibility.
    
    Returns
    -------
    np.ndarray, shape (num_replicates, 52)
    """
    rng = np.random.RandomState(rng_seed)
    n_features = len(deterministic)
    result = np.zeros((num_replicates, n_features))
    
    noise_fraction = pct / 100.0
    
    for i in range(num_replicates):
        if noise_fraction > 0:
            dev = deterministic * noise_fraction
            # Floor: allow small noise even for zero compartments
            dev[dev == 0] = 1.0
            noise = rng.randn(n_features) * dev
            noisy = deterministic + noise
        else:
            noisy = deterministic.copy()
        
        noisy = np.round(noisy)
        noisy[noisy < 0] = 0
        result[i] = noisy
    
    return result


def apply_uniform_noise(deterministic, pct, num_replicates, rng_seed=42):
    """
    Apply uniform redistribution noise to the deterministic baseline.
    
    For each timepoint block of 13 cell types:
    1. Round to integer cell counts
    2. Compute total cells at this timepoint
    3. Calculate n_move = round(total × pct / 100)
    4. Randomly remove n_move cells from occupied columns (weighted by count)
    5. Randomly add n_move cells to uniformly chosen columns
    
    Total cell count per timepoint is preserved.
    
    Parameters
    ----------
    deterministic : np.ndarray, shape (52,)
        Flattened deterministic baseline (4 timepoints × 13 cell types).
    pct : int
        Noise percentage (0-100).
    num_replicates : int
        Number of noisy replicates to generate.
    rng_seed : int
        Random seed for reproducibility.
    
    Returns
    -------
    np.ndarray, shape (num_replicates, 52)
    """
    rng = np.random.RandomState(rng_seed)
    n_features = len(deterministic)
    result = np.zeros((num_replicates, n_features))
    
    # Integer baseline
    det_int = np.round(deterministic).astype(int)
    det_int[det_int < 0] = 0
    
    for i in range(num_replicates):
        row = det_int.copy()
        
        for t in range(N_TIMEPOINTS):
            start = t * N_CELL_TYPES
            end = start + N_CELL_TYPES
            block = row[start:end].copy()
            
            total = int(np.sum(block))
            if total == 0:
                continue
            
            n_move = int(np.round(total * pct / 100.0))
            
            if n_move == 0:
                continue
            
            if n_move >= total:
                # 100% noise: completely random distribution
                new_indices = rng.randint(0, N_CELL_TYPES, size=total)
                row[start:end] = np.bincount(new_indices, minlength=N_CELL_TYPES)
                continue
            
            # Expand row into flat list of column indices
            # e.g., [3, 0, 5] -> [0, 0, 0, 2, 2, 2, 2, 2]
            flat_indices = np.repeat(np.arange(N_CELL_TYPES), block)
            
            # Shuffle to randomly select which cells to move
            rng.shuffle(flat_indices)
            
            # Keep first (total - n_move) cells in place
            n_keep = total - n_move
            kept = flat_indices[:n_keep]
            
            # Redistribute the moved cells to uniformly random columns
            new_positions = rng.randint(0, N_CELL_TYPES, size=n_move)
            
            # Reconstruct the row
            final = np.concatenate([kept, new_positions])
            row[start:end] = np.bincount(final, minlength=N_CELL_TYPES)
        
        result[i] = row
    
    return result


print("Functions defined successfully.")


Functions defined successfully.


## Generate All Noise Variants
Run the cell below to generate all noise variants for all 12 metaclones. This creates:
- 12 metaclones × 51 noise levels × 2 noise types = **1,224 CSV files**


In [3]:
# =========================================================
# Main Execution Loop
# =========================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_files = 0

for meta_id in range(1, 13):
    mc_dir_name = f'MC{meta_id - 1}'  # 0-indexed directory names
    num_replicates = REPLICATE_COUNTS[meta_id]
    
    print(f'\n{"="*60}')
    print(f'Processing Metaclone {meta_id} -> {mc_dir_name} ({num_replicates} replicates)')
    print(f'{"="*60}')
    
    # Load deterministic baseline from standard dataset
    baseline = load_deterministic_baseline(meta_id)
    print(f'  Baseline loaded: {len(baseline)} features, sum={baseline.sum():.0f}')
    
    for pct in NOISE_LEVELS:
        dev_folder = get_dev_folder(pct)
        out_dir = os.path.join(OUTPUT_DIR, mc_dir_name, dev_folder)
        os.makedirs(out_dir, exist_ok=True)
        
        # Use different seeds for Gaussian vs Uniform to ensure independence
        gaussian_seed = meta_id * 1000 + pct
        uniform_seed = meta_id * 1000 + pct + 500
        
        # --- Gaussian Noise ---
        gaussian_data = apply_gaussian_noise(baseline, pct, num_replicates, rng_seed=gaussian_seed)
        gaussian_path = os.path.join(out_dir, f'mc_{meta_id}_gaussian_{pct}.csv')
        np.savetxt(gaussian_path, gaussian_data, delimiter=',', fmt='%.4f')
        
        # --- Uniform Noise ---
        uniform_data = apply_uniform_noise(baseline, pct, num_replicates, rng_seed=uniform_seed)
        uniform_path = os.path.join(out_dir, f'mc_{meta_id}_uniform_{pct}.csv')
        np.savetxt(uniform_path, uniform_data, delimiter=',', fmt='%d')
        
        total_files += 2
    
    print(f'  Saved {len(NOISE_LEVELS) * 2} files ({len(NOISE_LEVELS)} Gaussian + {len(NOISE_LEVELS)} Uniform)')

print(f'\n{"="*60}')
print(f'All simulations complete!')
print(f'Total files generated: {total_files}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'{"="*60}')



Processing Metaclone 1 -> MC0 (15 replicates)
  Baseline loaded: 52 features, sum=809
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 2 -> MC1 (30 replicates)
  Baseline loaded: 52 features, sum=813
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 3 -> MC2 (50 replicates)
  Baseline loaded: 52 features, sum=803
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 4 -> MC3 (40 replicates)
  Baseline loaded: 52 features, sum=810
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 5 -> MC4 (72 replicates)
  Baseline loaded: 52 features, sum=811
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 6 -> MC5 (56 replicates)
  Baseline loaded: 52 features, sum=811
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 7 -> MC6 (88 replicates)
  Baseline loaded: 52 features, sum=812
  Saved 102 files (51 Gaussian + 51 Uniform)

Processing Metaclone 8 -> MC7 (20 replicates)
  Baseline loaded: 52 feature

## Verification
Quick sanity checks on the generated data.


In [4]:
# =========================================================
# Verification: Check a few files and properties
# =========================================================

import glob

print("Directory structure:")
for mc_idx in range(12):
    mc_dir = os.path.join(OUTPUT_DIR, f'MC{mc_idx}')
    if os.path.exists(mc_dir):
        subdirs = sorted(os.listdir(mc_dir))
        n_files = sum(len(os.listdir(os.path.join(mc_dir, sd))) for sd in subdirs)
        print(f"  MC{mc_idx}: {len(subdirs)} folders, {n_files} total files")

print()

# Check properties of Gaussian noise
meta_id = 1
baseline = load_deterministic_baseline(meta_id)
print(f"Metaclone {meta_id} baseline (first 13 values = timepoint 0):")
print(f"  {baseline[:13]}")

# Load 0% Gaussian (should match baseline)
g0 = np.loadtxt(os.path.join(OUTPUT_DIR, 'MC0', 'dev_0_19', 'mc_1_gaussian_0.csv'), delimiter=',')
print(f"\nGaussian 0% (first row): {g0[0, :13]}")
print(f"  All rows identical? {np.all(g0 == g0[0])}")

# Load 50% Gaussian
g50 = np.loadtxt(os.path.join(OUTPUT_DIR, 'MC0', 'dev_40_59', 'mc_1_gaussian_50.csv'), delimiter=',')
print(f"\nGaussian 50% mean (first 13): {np.mean(g50[:, :13], axis=0).round(1)}")
print(f"  Std dev (first 13): {np.std(g50[:, :13], axis=0).round(1)}")

# Check Uniform noise preserves totals
u0 = np.loadtxt(os.path.join(OUTPUT_DIR, 'MC0', 'dev_0_19', 'mc_1_uniform_0.csv'), delimiter=',')
u50 = np.loadtxt(os.path.join(OUTPUT_DIR, 'MC0', 'dev_40_59', 'mc_1_uniform_50.csv'), delimiter=',')
u100 = np.loadtxt(os.path.join(OUTPUT_DIR, 'MC0', 'dev_80_100', 'mc_1_uniform_100.csv'), delimiter=',')

print(f"\nUniform noise - total count preservation check (timepoint 0, first 13 cols):")
print(f"  0% total per row:   {u0[:5, :13].sum(axis=1)}")
print(f"  50% total per row:  {u50[:5, :13].sum(axis=1)}")
print(f"  100% total per row: {u100[:5, :13].sum(axis=1)}")

# Check 100% uniform is truly random
print(f"\nUniform 100% (first row, timepoint 0): {u100[0, :13]}")
print(f"  Sum: {u100[0, :13].sum()}")
print(f"  vs baseline sum: {baseline[:13].sum()}")

print("\nVerification complete!")


Directory structure:
  MC0: 5 folders, 102 total files
  MC1: 5 folders, 102 total files
  MC2: 5 folders, 102 total files
  MC3: 5 folders, 102 total files
  MC4: 5 folders, 102 total files
  MC5: 5 folders, 102 total files
  MC6: 5 folders, 102 total files
  MC7: 5 folders, 102 total files
  MC8: 5 folders, 102 total files
  MC9: 5 folders, 102 total files
  MC10: 5 folders, 102 total files
  MC11: 5 folders, 102 total files

Metaclone 1 baseline (first 13 values = timepoint 0):
  [11.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]

Gaussian 0% (first row): [11.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
  All rows identical? True

Gaussian 50% mean (first 13): [9.9 0.3 0.2 0.2 0.3 0.7 0.2 0.2 0.3 0.5 0.4 0.3 0.5]
  Std dev (first 13): [3.9 0.6 0.5 0.4 0.5 0.7 0.4 0.4 0.4 0.7 0.7 0.6 0.9]

Uniform noise - total count preservation check (timepoint 0, first 13 cols):
  0% total per row:   [11. 11. 11. 11. 11.]
  50% total per row:  [11. 11. 11. 11. 11.]
  100% total per row: [1